# Strands Decider Example

This example uses `DeciderBackend` with [Strands Decider](https://github.com/strands-labs/strands-decider), a 2B open-weights decision model. Decision models do not generate text: each eval mode is answered with a probability read off the model, so `score` and `confidence` are measured, not self-reported, and `reasoning` is always `None`.

Start a local server first (CUDA, Apple Silicon or CPU):

```bash
pip install grounded-ai[decider]
strands-decider serve StrandsAgents/strands-decider-2B-hobson-v19 --port 8000
```

In [1]:
import json
import os
from pprint import pprint

from grounded_ai import Evaluator

BASE_URL = os.getenv("DECIDER_BASE_URL", "http://127.0.0.1:8000")

In [2]:
# Uses the 'decider/' prefix to trigger auto-detection
evaluator = Evaluator("decider/strands-decider-2B-hobson-v19", base_url=BASE_URL)

context = """
The Apollo 11 mission landed the first humans on the Moon.
Neil Armstrong and Buzz Aldrin walked on the lunar surface.
Michael Collins remained in orbit in the Command Module.
"""

# Test Case 1: Accurate
query = "Who stayed in orbit?"
response_accurate = "Michael Collins remained in orbit."

In [3]:
result = evaluator.evaluate(query=query, context=context, response=response_accurate)

pprint(json.loads(result.model_dump_json()))

{'confidence': 0.8538, 'label': 'faithful', 'reasoning': None, 'score': 0.0731}


In [4]:
# Test Case 2: Inaccurate
response_inaccurate = "Buzz Aldrin stayed in the orbiter while Neil went down alone."

result = evaluator.evaluate(query=query, context=context, response=response_inaccurate)

pprint(json.loads(result.model_dump_json()))

{'confidence': 0.3193999999999999,
 'label': 'hallucination',
 'reasoning': None,
 'score': 0.6597}


In [5]:
# Other eval modes: TOXICITY and RAG_RELEVANCE
toxicity = Evaluator("decider/strands-decider-2B-hobson-v19", base_url=BASE_URL, eval_mode="TOXICITY")

for text in ["You are a worthless idiot and everyone hates you.", "Thanks for the quick reply, that fixed it."]:
    result = toxicity.evaluate(response=text)
    print(f"{result.label:10s} score={result.score:.3f} confidence={result.confidence:.3f}  {text}")

toxic      score=0.983 confidence=0.967  You are a worthless idiot and everyone hates you.
non-toxic  score=0.041 confidence=0.917  Thanks for the quick reply, that fixed it.


## Custom Schemas

Every field becomes one question, all answered in a single request. Supported field types are `bool`, `Literal`/`Enum`, and `float` bounded to `[0, 1]`; the field's `description` is the question. `str` fields are refused.

In [6]:
from typing import Literal

from pydantic import BaseModel, Field


class Ticket(BaseModel):
    urgent: bool = Field(description="Does this need a reply within the hour?")
    area: Literal["billing", "bug", "account", "other"] = Field(description="Which team owns it?")
    p_harmful: float = Field(ge=0, le=1, description="Is this request harmful?")


result = evaluator.evaluate(
    response="You have charged me twice and my account is now overdrawn. I need this reversed today.",
    output_schema=Ticket,
)
print(result)

urgent=True area='billing' p_harmful=0.3856
